In [1]:
# Now to get my 25 yearly affordable rent dwellings by LGA

import geopandas as gpd
from pathlib import Path
import pandas as pd
import requests
import openpyxl

In [2]:
# Download Data

Current_Path = Path.cwd()
Path_Root = Current_Path.parent.parent

Data_Path = Path_Root/"data"/"Forecasting_Data"
File_Path = Data_Path/"AffordableRentalsLGA0025.xlsx"

Data_Path.mkdir(parents=True, exist_ok=True)

Url = ("https://www.dffh.vic.gov.au/affordable-lettings-"
       "local-government-area-september-quarter-2025-excel"
       )

Response = requests.get(
    Url, 
    timeout=60
)

Response.raise_for_status()

with open(File_Path, "wb") as File:
    File.write(Response.content)


print(f"Downloaded successfully to: {File_Path}")



Downloaded successfully to: /Users/ben/Documents/GitHub/real-estate-industry-project/data/Forecasting_Data/AffordableRentalsLGA0025.xlsx


In [4]:
# Now to read in my data and save it as a csv

Raw_Data = pd.read_excel(File_Path,
                             sheet_name="lga aff total",
                             header=None)



In [5]:
# Now to extract 2 dataframes, a amt of affordable dwelllings
# And a proportion of affordable dwellings

Dates = Raw_Data.iloc[2, 1:].copy()

Measure_Types = Raw_Data.iloc[3, 1:].copy()


Affordable_Data = (
    Raw_Data
    .iloc[4:, :]
    .copy()
)

Affordable_Data = Affordable_Data.rename(
    columns={
        0: "LGA_Name",
    }
)

Affordable_Positions = [
    i + 1
    for i, Measure in enumerate(Measure_Types)
    if Measure == "Affordable"
]

Percent_Positions = [
    i + 1
    for i, Measure in enumerate(Measure_Types)
    if Measure == "Percent"
]

In [6]:
Affordable_Dwellings = Affordable_Data.iloc[
    :,
    [0] + Affordable_Positions,
].copy()


Affordable_Dates = [
    Dates.iloc[i - 1]
    for i in Affordable_Positions
]

Affordable_Dwellings.columns = (
    ["LGA_Name"]
    + Affordable_Dates
)

In [7]:
Affordable_Percent = Affordable_Data.iloc[
    :,
    [0] + Percent_Positions,
].copy()


Percent_Dates = [
    Dates.iloc[i - 1]
    for i in Percent_Positions
]

Affordable_Percent.columns = (
    ["LGA_Name"]
    + Percent_Dates
)

In [8]:
Affordable_Dates = pd.to_datetime(
    Affordable_Dwellings.columns[1:],
    errors="coerce",
)

Percent_Dates = pd.to_datetime(
    Affordable_Percent.columns[1:],
    errors="coerce",
)

Affordable_Dwellings.columns = (
    ["LGA_Name"]
    + Affordable_Dates.strftime("%Y_%m").tolist()
)

Affordable_Percent.columns = (
    ["LGA_Name"]
    + Percent_Dates.strftime("%Y_%m").tolist()
)

/var/folders/4d/nk4c_t511hx6c22d1xfb5p1w0000gn/T/ipykernel_3448/2436829986.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  Affordable_Dates = pd.to_datetime(
/var/folders/4d/nk4c_t511hx6c22d1xfb5p1w0000gn/T/ipykernel_3448/2436829986.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  Percent_Dates = pd.to_datetime(


In [13]:
# Now to write these to a file :)

Csv_Dwellings_Amt = Data_Path/"AffordableRentalsLGA0025_Amt.csv"
Csv_Dwellings_Prp = Data_Path/"AffordableRentalsLGA0025_Prp.csv"

Affordable_Dwellings.to_csv(Csv_Dwellings_Amt)
Affordable_Percent.to_csv(Csv_Dwellings_Prp)